# Policy Consolidation

This notebook demonstrates the consolidation process that groups policies by vehicle (VIN) and assigns unified superpolicy IDs.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import polars as pl

## Define file paths

Set input and output locations. Update these paths to match your environment.

In [ ]:
car_parquet_file = '/Users/Mach/dev/aps/data/2026_Dmodel_data/master_dataset_car.parquet'
output_parquet = '/Users/Mach/dev/aps/data/2026_Dmodel_data/superpolicy/superpolicy_car.parquet'

print(f"Input file: {car_parquet_file}")
print(f"Output file: {output_parquet}")

## Load data (lazy, select only needed columns)

Read only the columns needed for consolidation to minimize memory usage.

In [ ]:
# Read only required columns from parquet (lazy loading for efficiency)
df = pl.scan_parquet(car_parquet_file).select(['vin', 'vin_date', 'policyid']).collect()

print(f"Loaded {df.shape[0]:,} rows with columns: {df.columns}")

## Run consolidation

Execute the consolidation algorithm. This process:
1. Generates a row ID for superpolicy assignment
2. Assigns initial superpolicy IDs (minimum ID per policy)
3. Iteratively consolidates policies where the same VIN crosses multiple policies
4. Outputs the consolidated dataset

In [ ]:
max_iterations = 14

# Generate row ID for superpolicy assignment
df = df.with_row_index('ID')

# Initial assignment: minimum ID per policy
df = df.with_columns(
    pl.col('ID').min().over('policyid').alias('superpolicy_id')
)

print("Initial assignment complete")
print()

# Iterative consolidation
for iteration in range(max_iterations):
    iter_num = iteration + 1
    
    # Find VINs that cross multiple superpolicies
    crossing_vins = (
        df.group_by('vin')
        .agg(pl.col('superpolicy_id').n_unique().alias('policy_count'))
        .filter(pl.col('policy_count') > 1)
    )
    
    crossing_count = crossing_vins.shape[0]
    print(f"Iteration {iter_num:2d}: {crossing_count:,} VINs cross policies", end="")
    
    if crossing_count == 0:
        print(" → CONVERGED")
        break
    
    print()
    
    # Consolidate: assign minimum superpolicy_id across VIN, then across policyid
    df = df.with_columns(
        pl.col('superpolicy_id').min().over('vin').alias('superpolicy_id')
    )
    df = df.with_columns(
        pl.col('superpolicy_id').min().over('policyid').alias('superpolicy_id')
    )

print(f"\nConsolidation complete after {iter_num} iterations")

## Save output

Save only `vin_date` and `superpolicy_id` for easy merging with original data.

In [ ]:
import os

# Create output directory if it doesn't exist
os.makedirs(os.path.dirname(output_parquet), exist_ok=True)

# Select output columns and sort by ID before saving
output_df = df.sort('ID').select(['vin_date', 'superpolicy_id'])
output_df.write_parquet(output_parquet)

print(f"Output saved: {output_parquet}")
print(f"Output shape: {output_df.shape}")

## Display sample records

View the first and last few rows of the consolidated dataset.

In [ ]:
print("First 5 rows:")
print(output_df.head())

In [ ]:
print("\nLast 5 rows:")
print(output_df.tail())

## Summary statistics

Detailed analysis of the consolidation results.

In [ ]:
stats = {
    'Total Records': df.shape[0],
    'Original Policies': df['policyid'].n_unique(),
    'Consolidated Policies': df['superpolicy_id'].n_unique(),
    'Unique VINs': df['vin'].n_unique(),
    'Policy Reduction': f"{((df['policyid'].n_unique() - df['superpolicy_id'].n_unique()) / df['policyid'].n_unique() * 100):.2f}%"
}

for key, value in stats.items():
    print(f"{key:25s}: {value:>20}")

## Data quality verification

Confirm data integrity by checking for null values.

In [ ]:
null_check = {
    'ID': df['ID'].null_count(),
    'policyid': df['policyid'].null_count(),
    'vin': df['vin'].null_count(),
    'vin_date': df['vin_date'].null_count(),
    'superpolicy_id': df['superpolicy_id'].null_count()
}

print("Null values per column:")
for col, count in null_check.items():
    status = "✓" if count == 0 else "✗"
    print(f"  {status} {col}: {count}")

## Distribution analysis

Examine superpolicy size distribution.

In [ ]:
policy_sizes = df.group_by('superpolicy_id').agg(pl.len().alias('record_count'))

print("Superpolicy record count distribution:")
print(f"  Min:  {policy_sizes['record_count'].min():,}")
print(f"  Max:  {policy_sizes['record_count'].max():,}")
print(f"  Mean: {policy_sizes['record_count'].mean():.0f}")
print(f"  Median: {policy_sizes['record_count'].median():.0f}")